# Análisis de sesgos del corpus de recetas

Este cuaderno estudia posibles sesgos del corpus del proyecto RAG de recetas desde el contexto real del dominio: procedencia de las recetas, concentración temática, autores, ingredientes y posibles desbalances geográficos o de accesibilidad culinaria.

El objetivo no es demostrar que el corpus está "libre de sesgos" sino localizar concentraciones que puedan afectar al retrieval y a la generación, y proponer mitigaciones concretas para el proyecto.

## Qué sesgos se van a revisar

En este proyecto tiene sentido analizar principalmente: 
- Sesgo de origen o fuente: dominios o autores demasiado dominantes.
- Sesgo temático: recetas muy concentradas en unos pocos platos o ingredientes.
- Sesgo geográfico: presencia desigual de cocinas o lugares mencionados.
- Sesgo de accesibilidad: recetas que favorecen ingredientes caros o poco comunes.
- Sesgo de complejidad: exceso de recetas muy cortas o muy largas que distorsionen el retrieval.

Cuando el corpus no disponga de una columna explícita, el análisis se apoya en proxies razonables como `autor`, `titulo`, `ingredientes` e `instrucciones`.

In [25]:
from pathlib import Path
import json
import math
import re
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'ChefGPT_Dataset_Random_Sample.json').exists():
            return candidate
    return start

ROOT = find_repo_root(Path.cwd().resolve())
RAW_PATH = ROOT / 'ChefGPT_Dataset_Random_Sample.json'
PREPROCESSED_PATH = ROOT / 'data' / 'preprocesado' / 'ChefGPT_Dataset_Preprocesado.json'
DATA_PATH = RAW_PATH

print(f'ROOT: {ROOT}')
print(f'Dataset bruto: {RAW_PATH.exists()}')
print(f'Dataset preprocesado: {PREPROCESSED_PATH.exists()}')

ROOT: C:\Users\pc\Documents\MASTER\SEGUNDO SEMESTRE\NLP\Proyecto\ProyectoPLN
Dataset bruto: True
Dataset preprocesado: True


In [26]:
with DATA_PATH.open('r', encoding='utf-8') as handle:
    raw_recipes = json.load(handle)

df = pd.DataFrame(raw_recipes)
print(f'Registros: {len(df)}')
print(f'Columnas: {list(df.columns)}')
df.head(3)

Registros: 1500
Columnas: ['titulo', 'ingredientes', 'instrucciones', 'tiempo_total', 'porciones', 'autor', 'n_ingredientes']


,titulo,ingredientes,instrucciones,tiempo_total,porciones,autor,n_ingredientes
0,"Empanadillas de atún, tomate y huevo: un clási...","[300g Atún en conserva, 2 Huevo, Salsa de toma...",comenzamos cociendo los huevos en agua con sal...,15 minutos,4,Pakus,4
1,"Receta de caballa con salsa de tomate, la rece...","[2 Caballa, 400ml Salsa de tomate, 2 Patata, A...","en una cazuela baja, freímos ligeramente los l...",30 minutos,2,Pakus,6
2,Las naranjas están en su mejor momento: las me...,"[1 Pollo entero, 3 Naranja, 2 Patata, Finas Hi...","limpiamos el pollo de plumones, vigilad en esp...",90 minutos,4,Pakus,7


In [42]:
def first_existing(columns, candidates):
    for candidate in candidates:
        if candidate in columns:
            return candidate
    return None

author_col = first_existing(df.columns, ['autor', 'author', 'by', 'creator'])

print('author_col =', author_col)

def concentration_metrics(series: pd.Series) -> pd.DataFrame:
    counts = series.dropna().astype(str).str.strip()
    counts = counts[counts != '']
    if counts.empty:
        return pd.DataFrame([{'n_values': 0, 'top_share': np.nan, 'hhi': np.nan, 'entropy': np.nan}])
    freq = counts.value_counts()
    shares = freq / freq.sum()
    return pd.DataFrame([{
        'n_values': int(freq.shape[0]),
        'top_share': float(shares.iloc[0]),
        'hhi': float((shares ** 2).sum()),
        'entropy': float(-(shares * np.log2(shares)).sum()),
    }])

if author_col:
    author_stats = concentration_metrics(df[author_col])
    display(author_stats)
    display(df[author_col].fillna('Sin autor').astype(str).str.strip().replace('', 'Sin autor').value_counts().head(10))

author_col = autor


,n_values,top_share,hhi,entropy
0,26,0.180667,0.109731,3.591195


autor
Liliana Fuchs                           271
Pakus                                   240
Carmen Tía Alia                         226
Esther Clemente                         146
Maria Jose                              139
Mesa Cero Chefs y Jaime de las Heras     89
Inés Vazquez Noya                        76
Unodedos                                 55
Marina Blanco                            47
Miguel Ayuso Rejas                       38
Name: count, dtype: int64

## TF-IDF sobre el corpus

El objetivo aquí es detectar vocabulario dominante y ver si el corpus se concentra demasiado en unos pocos ingredientes, técnicas o regiones culinarias. Eso ayuda a identificar sesgos temáticos y de accesibilidad.

In [44]:
SPANISH_STOPWORDS = {
    'a', 'acerca', 'ademas', 'ahi', 'al', 'algo', 'algunas', 'algunos', 'ante', 'antes', 'asi', 'aun',
    'bajo', 'bien', 'cada', 'como', 'con', 'contra', 'cual', 'cuando', 'de', 'del', 'desde', 'donde',
    'e', 'el', 'ella', 'ellas', 'ellos', 'en', 'entre', 'era', 'erais', 'eran', 'eras', 'eres', 'es',
    'esa', 'esas', 'ese', 'esos', 'esta', 'estaba', 'estaban', 'estado', 'estais', 'estamos', 'estan',
    'estar', 'este', 'esto', 'estos', 'fue', 'fueron', 'ha', 'hace', 'hacer', 'han', 'hasta', 'hay',
    'la', 'las', 'lo', 'los', 'mas', 'me', 'mi', 'mis', 'mientras', 'muy', 'no', 'nos', 'nosotros',
    'o', 'os', 'otra', 'otros', 'para', 'pero', 'poco', 'por', 'porque', 'que', 'quien', 'quienes',
    'se', 'sea', 'si', 'sin', 'sobre', 'son', 'su', 'sus', 'tambien', 'te', 'tenia', 'tenian', 'tiene',
    'toda', 'todas', 'todo', 'todos', 'tu', 'tus', 'un', 'una', 'unas', 'uno', 'unos', 'ya',
}
TOKEN_RE = re.compile(r'[a-záéíóúñü]+', re.IGNORECASE)


def normalize_text(text):
    if text is None:
        return ''
    text = str(text).lower()
    text = text.replace('\n', ' ').replace('\r', ' ')
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def tokenize(text):
    tokens = TOKEN_RE.findall(normalize_text(text))
    return [token for token in tokens if len(token) > 2 and token not in SPANISH_STOPWORDS]


text_columns = [col for col in ['titulo', 'ingredientes', 'instrucciones'] if col in df.columns]
df['analysis_text'] = df[text_columns].fillna('').astype(str).agg(' '.join, axis=1)
df['tokens'] = df['analysis_text'].apply(tokenize)

doc_term_counts = [Counter(tokens) for tokens in df['tokens']]
doc_freq = Counter()
for tokens in df['tokens']:
    doc_freq.update(set(tokens))

n_docs = len(df)
idf = {term: math.log((1 + n_docs) / (1 + freq)) + 1 for term, freq in doc_freq.items()}


def top_tfidf_terms_for_indices(indices, top_n=12):
    aggregate = Counter()
    total_tokens = 0
    for idx in indices:
        counts = doc_term_counts[idx]
        aggregate.update(counts)
        total_tokens += sum(counts.values())
    if total_tokens == 0:
        return []
    scores = {}
    for term, count in aggregate.items():
        tf = count / total_tokens
        scores[term] = tf * idf.get(term, 0.0)
    return sorted(scores.items(), key=lambda item: item[1], reverse=True)[:top_n]


overall_terms = top_tfidf_terms_for_indices(range(len(df)), top_n=20)
overall_terms_df = pd.DataFrame(overall_terms, columns=['termino', 'tfidf'])
display(overall_terms_df)


def group_tfidf_report(group_col, min_docs=10, top_groups=8):
    rows = []
    if group_col is None:
        return pd.DataFrame(rows)
    grouped = df.groupby(group_col).indices
    ranked_groups = sorted(grouped.items(), key=lambda item: len(item[1]), reverse=True)
    for group_value, indices in ranked_groups[:top_groups]:
        if len(indices) < min_docs:
            continue
        terms = top_tfidf_terms_for_indices(indices, top_n=8)
        rows.append({
            'grupo': str(group_value),
            'n_docs': len(indices),
            'top_terms': ', '.join([term for term, _ in terms]),
        })
    return pd.DataFrame(rows)

if author_col:
    display(group_tfidf_report(author_col))

,termino,tfidf
0,aceite,0.017484
1,minutos,0.016179
2,azúcar,0.015928
3,sal,0.014235
4,salsa,0.013725
5,queso,0.013535
6,agua,0.013168
7,horno,0.012911
8,más,0.012761
9,harina,0.012377


,grupo,n_docs,top_terms
0,Liliana Fuchs,271,"añadir, dejar, más, aceite, sal, azúcar, minut..."
1,Pakus,240,"salsa, pollo, carne, caldo, aceite, tomate, mi..."
2,Carmen Tía Alia,226,"queso, minutos, aceite, retiramos, durante, re..."
3,Esther Clemente,146,"azúcar, añadimos, crema, comenzaremos, horno, ..."
4,Maria Jose,139,"vel, azúcar, masa, thermomix, ponemos, harina,..."
5,Mesa Cero Chefs y Jaime de las Heras,89,"minutos, aceite, harina, masa, sal, agua, azúc..."
6,Inés Vazquez Noya,76,"aceite, pimienta, oliva, sal, cocinar, cebolla..."
7,Unodedos,55,"azúcar, masa, horno, echamos, leche, crema, he..."


**Interpretación de resultados:**

Los términos más relevantes del TF-IDF apuntan a un corpus muy centrado en cocina cotidiana y de uso general. Aparecen con fuerza palabras como `aceite`, `sal`, `agua`, `horno`, `harina`, `masa`, `pollo` o `cebolla`, lo que sugiere que el corpus está dominado por recetas domésticas, ingredientes comunes y técnicas básicas de preparación.

Esto no es un problema en sí, pero sí nos dice que el corpus puede estar sesgado hacia recetas populares y accesibles, con menor presencia de platos más especializados, regionales o de alta complejidad. En otras palabras, el sistema RAG tenderá a recuperar primero recetas muy generales y frecuentes, y podría dejar menos representadas cocinas minoritarias, ingredientes menos comunes o estilos culinarios más específicos.

Si en los grupos por autor o fuente aparecen términos distintos, eso indicaría además una concentración temática por procedencia: cada autor podría estar repitiendo su propio vocabulario o estilo culinario. En conjunto, el TF-IDF ayuda a ver que el sesgo principal aquí no parece demográfico, sino de cobertura del dominio y de repetición de patrones culinarios muy frecuentes.

## Topic modeling con LDA

Esta celda extrae temas latentes y ayuda a ver si el corpus está excesivamente concentrado en pocos patrones.

In [29]:
try:
    from sklearn.decomposition import LatentDirichletAllocation
    from sklearn.feature_extraction.text import CountVectorizer
except ImportError as exc:
    print(f'se omite LDA porque falta scikit-learn: {exc}')
else:
    vectorizer = CountVectorizer(
        max_df=0.85,
        min_df=2,
        ngram_range=(1, 2),
        stop_words=list(SPANISH_STOPWORDS),
    )
    X = vectorizer.fit_transform(df['analysis_text'])
    n_topics = min(6, max(2, X.shape[0] // 50))
    lda = LatentDirichletAllocation(n_components=n_topics, random_state=42, learning_method='batch')
    topic_matrix = lda.fit_transform(X)
    feature_names = np.array(vectorizer.get_feature_names_out())

    def top_words_for_topic(topic_idx, top_n=10):
        component = lda.components_[topic_idx]
        return feature_names[np.argsort(component)[-top_n:][::-1]].tolist()

    topic_rows = []
    for topic_idx in range(n_topics):
        topic_rows.append({
            'topic_id': topic_idx,
            'top_words': ', '.join(top_words_for_topic(topic_idx, top_n=10)),
            'peso_medio': float(topic_matrix[:, topic_idx].mean()),
        })

    topics_df = pd.DataFrame(topic_rows)
    display(topics_df)

    dominant_topic = topic_matrix.argmax(axis=1)
    df['dominant_topic'] = dominant_topic
    topic_share = df['dominant_topic'].value_counts(normalize=True).sort_index()
    display(topic_share.to_frame('share'))

    if author_col:
        topic_by_author = pd.crosstab(df[author_col].fillna('Sin autor'), df['dominant_topic'])
        display(topic_by_author.head(10))
    if source_col:
        topic_by_source = pd.crosstab(df[source_col].fillna('Sin fuente'), df['dominant_topic'])
        display(topic_by_source.head(10))

,topic_id,top_words,peso_medio
0,0,"crema, receta, azúcar, queso, chocolate, nata,...",0.108740
1,1,"azúcar, masa, harina, horno, mantequilla, minu...",0.212214
2,2,"aceite, oliva, sal, aceite oliva, pimienta, mi...",0.155997
3,3,"azúcar, leche, dejamos, vaso, receta, ponemos,...",0.076559
4,4,"aceite, minutos, salsa, sal, fuego, agua, rece...",0.255478
5,5,"minutos, sal, aceite, salsa, receta, 1count, p...",0.191012


,share
dominant_topic,
0,0.098667
1,0.213333
2,0.149333
3,0.068667
4,0.283333
5,0.186667


dominant_topic,0,1,2,3,4,5
autor,,,,,,
Anabel Palomares,0,0,0,0,1,0
Bea Orviz Tjiang,4,1,3,0,18,3
Carmen Tía Alia,34,40,9,19,44,80
Esther Clemente,29,53,10,6,28,20
Inés Vazquez Noya,7,8,24,1,15,21
Jaime de las Heras,2,0,2,1,8,3
Jesús Rojas,0,0,0,1,0,2
Juana Trujillo,0,0,0,0,3,1
Lena Álvarez,0,1,1,0,0,0


**Interpretación cualitativa del LDA:**

Los temas del LDA suelen confirmar que el corpus no está organizado en una gran variedad de dominios culinarios, sino en unos pocos bloques recurrentes de vocabulario. Cuando varios temas comparten palabras como `aceite`, `sal`, `horno`, `pollo`, `harina` o `cebolla`, eso indica que el corpus repite con mucha frecuencia el mismo tipo de recetas y técnicas, en vez de repartir el contenido entre cocinas o estilos muy distintos.

Desde el punto de vista de sesgos, esto sugiere un sesgo de cobertura y de concentración temática. El RAG tenderá a aprender y recuperar mejor las recetas más repetidas, rápidas o genéricas, mientras que las recetas menos frecuentes, más regionales o más complejas quedan diluidas en temas residuales. Si además uno de los temas domina claramente en proporción, el sesgo es todavía más claro: el corpus está sobreponderado hacia una familia concreta de preparaciones.

En resumen, el LDA no apunta tanto a un sesgo por autoría individual como a una falta de diversidad temática real. Eso es importante porque, aunque el sistema funcione bien sobre recetas comunes, puede responder peor cuando el usuario pida cocina regional, platos menos habituales o alternativas que se salgan del patrón dominante del corpus.

## Hallazgos y mitigaciones

Con este tipo de corpus, los sesgos más probables no son demográficos en sentido estricto, sino de cobertura del dominio. Los problemas más importantes para un RAG son: 
- que unas pocas fuentes o autores dominen el índice,
- que ciertas cocinas o ingredientes aparezcan de forma desigual,
- que el sistema aprenda a recomendar solo recetas de un subgrupo muy repetido,
- que el retrieval devuelva casi siempre ejemplos muy similares y poco diversos.

Las mitigaciones deberían quedar reflejadas tanto en la memoria como en el pipeline.

In [31]:
mitigation_rows = [
    {
        'sesgo': 'Concentracion por autor o fuente',
        'evidencia': 'Muchas recetas proceden de pocas fuentes o autores si la distribucion presenta una cola muy larga.',
        'impacto': 'El RAG recupera siempre recetas del mismo origen y reduce diversidad.',
        'mitigacion': 'Añadir limites por fuente en scraping, equilibrar muestras por dominio y diversificar top-k con re-ranking.',
    },
    {
        'sesgo': 'Concentracion tematica',
        'evidencia': 'TF-IDF/LDA muestran unos pocos temas o ingredientes dominantes.',
        'impacto': 'El sistema favorece recetas muy parecidas y cubre peor platos menos frecuentes.',
        'mitigacion': 'Etiquetar tema/cocina, estratificar el muestreo y usar diversidad en retrieval.',
    },
    {
        'sesgo': 'Sesgo geografico o culinario',
        'evidencia': 'NER y analisis de palabras clave pueden mostrar una sola region o pais dominante.',
        'impacto': 'El asistente puede dar una vision reducida de la cocina disponible.',
        'mitigacion': 'Ampliar las fuentes a distintas regiones, añadir metadatos de origen y evaluar por subcorpus geografico.',
    },
    {
        'sesgo': 'Accesibilidad de ingredientes',
        'evidencia': 'Predominan ingredientes muy comunes o, al contrario, recetas con productos poco accesibles.',
        'impacto': 'La utilidad baja para usuarios con presupuesto o despensa limitada.',
        'mitigacion': 'Marcar nivel de dificultad y coste aproximado, y recuperar alternativas mas economicas.',
    },
    {
        'sesgo': 'Poca diversidad en el retrieval',
        'evidencia': 'El mismo tipo de receta aparece repetidamente en los primeros resultados.',
        'impacto': 'Se degrada la experiencia conversacional y se refuerzan patrones repetitivos.',
        'mitigacion': 'Aplicar deduplicacion por similitud, MMR o re-ranking con penalizacion de redundancia.',
    },
    {
        'sesgo': 'Evaluacion no estratificada',
        'evidencia': 'Las metricas globales esconden fallos en grupos minoritarios.',
        'impacto': 'Se sobreestima la calidad real del sistema.',
        'mitigacion': 'Reportar Recall@k, MRR o juicio humano por subgrupos: fuente, tema, region y coste.',
    },
]

mitigation_df = pd.DataFrame(mitigation_rows)
display(mitigation_df)

,sesgo,evidencia,impacto,mitigacion
0,Concentracion por autor o fuente,Muchas recetas proceden de pocas fuentes o aut...,El RAG recupera siempre recetas del mismo orig...,"Añadir limites por fuente en scraping, equilib..."
1,Concentracion tematica,TF-IDF/LDA muestran unos pocos temas o ingredi...,El sistema favorece recetas muy parecidas y cu...,"Etiquetar tema/cocina, estratificar el muestre..."
2,Sesgo geografico o culinario,NER y analisis de palabras clave pueden mostra...,El asistente puede dar una vision reducida de ...,"Ampliar las fuentes a distintas regiones, añad..."
3,Accesibilidad de ingredientes,"Predominan ingredientes muy comunes o, al cont...",La utilidad baja para usuarios con presupuesto...,"Marcar nivel de dificultad y coste aproximado,..."
4,Poca diversidad en el retrieval,El mismo tipo de receta aparece repetidamente ...,Se degrada la experiencia conversacional y se ...,"Aplicar deduplicacion por similitud, MMR o re-..."
5,Evaluacion no estratificada,Las metricas globales esconden fallos en grupo...,Se sobreestima la calidad real del sistema.,"Reportar Recall@k, MRR o juicio humano por sub..."


## Cómo documentarlo en el proyecto

Este cuaderno debe citarse en la memoria del proyecto dentro del apartado de identificación de sesgos. Además, conviene recoger las medidas en dos sitios más:
- `README.md`: una nota breve sobre qué sesgos se inspeccionan y qué mitigaciones aplica el RAG.
- la memoria final: una tabla con sesgo, evidencia, impacto y mitigación, preferiblemente junto al apartado de metodología.

Si se implementa una interfaz tipo Streamlit, también es útil exponer metadatos de fuente, tema y similitud para que el usuario vea de dónde sale cada respuesta y no perciba el sistema como una caja negra.

# Analisis de sesgos del corpus

Este cuaderno estudia posibles sesgos del corpus de recetas desde la perspectiva del proyecto de RAG. El objetivo no es medir sesgos demograficos personales, sino detectar asimetrias observables en el corpus que puedan afectar a la recuperacion y a la generacion: concentracion de fuentes, variedad tematica, sesgo geografico y sesgo hacia determinados estilos de receta.

Preguntas guia:
- ¿El corpus depende demasiado de unos pocos autores o dominios?
- ¿Hay vocabulario o tematicas dominantes que puedan sesgar el RAG hacia una cocina concreta?
- ¿Aparecen entidades o referencias geograficas concentradas en unas pocas zonas?
- ¿Las recetas mas frecuentes son las mas rapidas, las mas elaboradas o las mas accesibles?
- Si aparecen sesgos, ¿como se pueden mitigar en la recopilacion, el preprocesamiento y la recuperacion?

In [32]:
from __future__ import annotations

import json
import math
import re
import unicodedata
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd

ROOT = Path.cwd()
for candidate in [ROOT, ROOT.parent, ROOT.parent.parent]:
    if (candidate / 'ChefGPT_Dataset_Random_Sample.json').exists():
        ROOT = candidate
        break

RAW_PATH = ROOT / 'ChefGPT_Dataset_Random_Sample.json'
PREPROCESSED_PATH = ROOT / 'data' / 'preprocesado' / 'ChefGPT_Dataset_Preprocesado.json'
OUTPUT_DIR = ROOT / 'data' / 'sesgos'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = PREPROCESSED_PATH if PREPROCESSED_PATH.exists() else RAW_PATH
with DATA_PATH.open('r', encoding='utf-8') as handle:
    data = json.load(handle)

df = pd.DataFrame(data)
print(f'Corpus cargado desde: {DATA_PATH}')
print(f'Registros: {len(df):,}')

Corpus cargado desde: c:\Users\pc\Documents\MASTER\SEGUNDO SEMESTRE\NLP\Proyecto\ProyectoPLN\data\preprocesado\ChefGPT_Dataset_Preprocesado.json
Registros: 1,500


## Inventario del esquema

Antes de medir sesgos conviene ver que metadatos existen realmente. En este proyecto nos interesan sobre todo los campos que permitan analizar procedencia, autorias, tipo de receta y nivel de complejidad.

In [33]:
print('Columnas disponibles:')
for col in df.columns:
    print('-', col)

def first_existing(columns: Iterable[str]) -> Optional[str]:
    for col in columns:
        if col in df.columns:
            return col
    return None

author_col = first_existing(['autor', 'author'])
source_col = first_existing(['url', 'source', 'source_url', 'dominio', 'website'])
category_col = first_existing(['categoria', 'categoría', 'tipo_plato', 'tipo', 'region', 'país', 'pais', 'cocina'])
time_col = first_existing(['tiempo_min', 'tiempo_total', 'tiempo'])
portion_col = first_existing(['porciones_num', 'porciones'])

print('Campos candidatos detectados:')
print('author_col =', author_col)
print('source_col =', source_col)
print('category_col =', category_col)
print('time_col =', time_col)
print('portion_col =', portion_col)

def concentration_metrics(series: pd.Series) -> pd.DataFrame:
    counts = series.dropna().astype(str).str.strip()
    counts = counts[counts != '']
    if counts.empty:
        return pd.DataFrame([{'n_values': 0, 'top_share': np.nan, 'hhi': np.nan, 'entropy': np.nan}])
    freq = counts.value_counts()
    shares = freq / freq.sum()
    return pd.DataFrame([{
        'n_values': int(freq.shape[0]),
        'top_share': float(shares.iloc[0]),
        'hhi': float((shares ** 2).sum()),
        'entropy': float(-(shares * np.log2(shares)).sum()),
    }])

if author_col:
    author_stats = concentration_metrics(df[author_col])
    display(author_stats)
    display(df[author_col].fillna('Sin autor').astype(str).str.strip().replace('', 'Sin autor').value_counts().head(10))

if source_col:
    source_stats = concentration_metrics(df[source_col])
    display(source_stats)
    display(df[source_col].fillna('Sin fuente').astype(str).str.strip().replace('', 'Sin fuente').value_counts().head(10))

if category_col:
    category_stats = concentration_metrics(df[category_col])
    display(category_stats)
    display(df[category_col].fillna('Sin categoria').astype(str).str.strip().replace('', 'Sin categoria').value_counts().head(10))

Columnas disponibles:
- titulo
- ingredientes
- instrucciones
- tiempo_total
- porciones
- autor
- n_ingredientes
- tiempo_total_raw
- tiempo_min
- tiempo_total_norm
- porciones_raw
- porciones_num
- porciones_min
- porciones_max
- porciones_norm
- ingredientes_raw
- ingredientes_struct
- ingredientes_norm
- ingredientes_count_real
- n_ingredientes_raw
Campos candidatos detectados:
author_col = autor
source_col = None
category_col = None
time_col = tiempo_min
portion_col = porciones_num


,n_values,top_share,hhi,entropy
0,26,0.180667,0.109731,3.591195


autor
Liliana Fuchs                           271
Pakus                                   240
Carmen Tía Alia                         226
Esther Clemente                         146
Maria Jose                              139
Mesa Cero Chefs y Jaime de las Heras     89
Inés Vazquez Noya                        76
Unodedos                                 55
Marina Blanco                            47
Miguel Ayuso Rejas                       38
Name: count, dtype: int64

## Concentracion por autor o fuente

Un sesgo clasico en corpus web es la sobre-representacion de unos pocos autores o dominios. Si el RAG recupera sobre todo recetas de una sola procedencia, la diversidad de respuestas cae y el sistema puede heredar el estilo editorial de esa fuente.

Esta seccion calcula una medida simple de concentracion y muestra el reparto principal.

In [34]:
def normalize_text(value: Any) -> str:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return ''
    text = unicodedata.normalize('NFKD', str(value))
    text = ''.join(ch for ch in text if unicodedata.category(ch) != 'Mn')
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def gini_from_counts(counts: Iterable[int]) -> float:
    values = np.array([v for v in counts if v > 0], dtype=float)
    if len(values) == 0:
        return 0.0
    values = np.sort(values)
    n = len(values)
    cumulative = np.cumsum(values)
    return float((n + 1 - 2 * np.sum(cumulative) / cumulative[-1]) / n)

def hhi_from_counts(counts: Iterable[int]) -> float:
    values = np.array([v for v in counts if v > 0], dtype=float)
    if len(values) == 0:
        return 0.0
    shares = values / values.sum()
    return float(np.sum(shares ** 2))

def top_concentration_table(series: pd.Series, top_n: int = 10) -> pd.DataFrame:
    cleaned = series.fillna('').map(normalize_text)
    cleaned = cleaned[cleaned.astype(bool)]
    counts = cleaned.value_counts()
    if counts.empty:
        return pd.DataFrame(columns=['valor', 'recetas', 'share'])
    total = int(counts.sum())
    rows = []
    for value, count in counts.head(top_n).items():
        rows.append({'valor': value, 'recetas': int(count), 'share': round(count / total, 4)})
    return pd.DataFrame(rows)

if author_col is not None:
    author_counts = df[author_col].fillna('').map(normalize_text)
    author_counts = author_counts[author_counts.astype(bool)].value_counts()
    print('Autor(es) mas frecuentes:')
    display(top_concentration_table(df[author_col], top_n=10))
    print(f'Gini autores: {gini_from_counts(author_counts.values):.3f}')
    print(f'HHI autores: {hhi_from_counts(author_counts.values):.3f}')
else:
    print('No se encontro un campo de autor utilizable.')

if source_col is not None:
    def extract_domain(value: Any) -> str:
        text = normalize_text(value)
        if not text:
            return ''
        text = re.sub(r'^https?://', '', text)
        text = text.split('/')[0] if '/' in text else text
        text = text.split('?')[0]
        return text

    source_series = df[source_col].map(extract_domain if 'url' in source_col.lower() else normalize_text)
    source_series = source_series[source_series.astype(bool)]
    source_counts = source_series.value_counts()
    print('Fuentes / dominios mas frecuentes:')
    display(top_concentration_table(source_series, top_n=10))
    print(f'Gini fuentes: {gini_from_counts(source_counts.values):.3f}')
    print(f'HHI fuentes: {hhi_from_counts(source_counts.values):.3f}')
else:
    print('No se encontro un campo de fuente o URL utilizable.')

Autor(es) mas frecuentes:


,valor,recetas,share
0,liliana fuchs,271,0.1807
1,pakus,240,0.1600
2,carmen tia alia,226,0.1507
3,esther clemente,146,0.0973
4,maria jose,139,0.0927
5,mesa cero chefs y jaime de las heras,89,0.0593
6,ines vazquez noya,76,0.0507
7,unodedos,55,0.0367
8,marina blanco,47,0.0313
9,miguel ayuso rejas,38,0.0253


Gini autores: 0.655
HHI autores: 0.110
No se encontro un campo de fuente o URL utilizable.


## Corpus textual para TF-IDF

Para analizar sesgos lexicales combinamos titulo, ingredientes e instrucciones. Esto permite ver si el corpus se orienta demasiado hacia ciertos ingredientes, tecnicas, regiones o estilos de cocina.

La implementacion siguiente no depende de `scikit-learn`: calcula TF-IDF de forma manual para que el cuaderno siga siendo ejecutable con las dependencias del proyecto.

In [35]:
SPANISH_STOPWORDS = {
    'a', 'acerca', 'ahora', 'al', 'algo', 'algunas', 'algunos', 'ante', 'antes', 'como', 'con', 'contra',
    'cual', 'cuando', 'de', 'del', 'desde', 'donde', 'dos', 'el', 'ella', 'ellas', 'ellos', 'en', 'entre',
    'era', 'eramos', 'esa', 'esas', 'ese', 'eso', 'esos', 'esta', 'estaba', 'estamos', 'estan', 'este',
    'esto', 'estos', 'fue', 'ha', 'hace', 'hacer', 'hasta', 'hay', 'la', 'las', 'le', 'les', 'lo', 'los',
    'mas', 'mi', 'mis', 'muy', 'no', 'nos', 'nosotros', 'o', 'para', 'pero', 'por', 'porque', 'que', 'quien',
    'se', 'sin', 'sobre', 'su', 'sus', 'tambien', 'te', 'tiene', 'tienen', 'todo', 'tu', 'un', 'una', 'unas',
    'uno', 'unos', 'y', 'ya', 'con', 'debe', 'dar', 'dos', 'donde', 'durante', 'e', 'el', 'ella', 'ellas',
    'ellos', 'esta', 'esto', 'este', 'estos', 'esta', 'estas', 'ser', 'son', 'era', 'eran'
}

TOKEN_RE = re.compile(r'[a-zA-ZáéíóúüñÁÉÍÓÚÜÑ]{3,}')

def recipe_text(row: pd.Series) -> str:
    pieces = []
    for col in ['titulo', 'ingredientes', 'instrucciones']:
        if col in row and row[col] is not None:
            value = row[col]
            if isinstance(value, list):
                pieces.extend([str(x) for x in value])
            else:
                pieces.append(str(value))
    return ' '.join(pieces)

def tokenize(text: str) -> List[str]:
    text = unicodedata.normalize('NFKD', str(text).lower())
    text = ''.join(ch for ch in text if unicodedata.category(ch) != 'Mn')
    tokens = TOKEN_RE.findall(text)
    return [tok for tok in tokens if tok not in SPANISH_STOPWORDS]

corpus = df.apply(recipe_text, axis=1).fillna('').tolist()
tokenized_docs = [tokenize(text) for text in corpus]
doc_count = len(tokenized_docs)
doc_freq = Counter()
for tokens in tokenized_docs:
    doc_freq.update(set(tokens))

idf = {term: math.log((1 + doc_count) / (1 + dfreq)) + 1.0 for term, dfreq in doc_freq.items()}

def top_tfidf_terms(tokens: List[str], top_n: int = 15) -> List[Tuple[str, float]]:
    if not tokens:
        return []
    term_freq = Counter(tokens)
    length = len(tokens)
    scores = {term: (count / length) * idf.get(term, 0.0) for term, count in term_freq.items()}
    return sorted(scores.items(), key=lambda item: item[1], reverse=True)[:top_n]

overall_freq = Counter()
for tokens in tokenized_docs:
    overall_freq.update(tokens)

overall_tfidf = Counter()
for tokens in tokenized_docs:
    for term, score in top_tfidf_terms(tokens, top_n=30):
        overall_tfidf[term] += score

top_terms_overall = pd.DataFrame([
    {'term': term, 'freq': int(overall_freq[term]), 'avg_tfidf_like': round(score / doc_count, 4)}
    for term, score in overall_tfidf.most_common(20)
])
display(top_terms_overall)

,term,freq,avg_tfidf_like
0,azucar,1425,0.0114
1,queso,1026,0.0111
2,pollo,691,0.0104
3,salsa,1148,0.0096
4,leche,835,0.0092
5,chocolate,496,0.0081
6,masa,979,0.0080
7,cebolla,904,0.0078
8,mantequilla,956,0.0077
9,arroz,494,0.0073


## Analisis tematico por grupos

En lugar de un LDA completo, esta seccion calcula terminos discriminativos por grupo usando una variante simple de c-TF-IDF. Es util para detectar si ciertos grupos concentran una narrativa culinaria muy especifica: por ejemplo, recetas rapidas frente a elaboradas, o una fuente dominante frente a fuentes minoritarias.

Si el corpus incluye una categoria o una fuente clara, se usa ese campo. Si no, se crean grupos por complejidad temporal.

In [36]:
def length_bucket(minutes: Any) -> str:
    try:
        value = float(minutes)
    except Exception:
        return 'desconocido'
    if value <= 20:
        return 'muy_rapida'
    if value <= 45:
        return 'rapida'
    if value <= 90:
        return 'media'
    return 'larga'

group_col = category_col or source_col or author_col
if group_col is not None:
    groups = df[group_col].fillna('').map(normalize_text)
    groups = groups.where(groups.astype(bool), other='desconocido')
else:
    if time_col is not None:
        groups = df[time_col].map(length_bucket)
        group_col = 'bucket_tiempo'
    else:
        groups = pd.Series(['desconocido'] * len(df), index=df.index)
        group_col = 'grupo'

group_to_docs: Dict[str, List[List[str]]] = defaultdict(list)
for group, tokens in zip(groups.tolist(), tokenized_docs):
    group_to_docs[str(group)].append(tokens)

group_scores: List[Dict[str, Any]] = []
for group, docs_tokens in sorted(group_to_docs.items(), key=lambda item: len(item[1]), reverse=True):
    group_token_counter = Counter()
    for tokens in docs_tokens:
        group_token_counter.update(tokens)
    if not group_token_counter:
        continue
    group_size = sum(group_token_counter.values())
    term_scores = {}
    for term, count in group_token_counter.items():
        tf = count / group_size
        term_scores[term] = tf * idf.get(term, 0.0)
    top_terms = sorted(term_scores.items(), key=lambda item: item[1], reverse=True)[:12]
    for rank, (term, score) in enumerate(top_terms, start=1):
        group_scores.append({
            'grupo': group,
            'n_recetas': len(docs_tokens),
            'termino': term,
            'score': round(score, 5),
            'rank': rank,
        })

topic_table = pd.DataFrame(group_scores)
if not topic_table.empty:
    display(topic_table.head(36))
else:
    print('No se pudieron construir grupos tematicos.')

,grupo,n_recetas,termino,score,rank
0,liliana fuchs,271,anadir,0.02487,1
1,liliana fuchs,271,poco,0.02037,2
2,liliana fuchs,271,dejar,0.02029,3
3,liliana fuchs,271,aceite,0.01953,4
4,liliana fuchs,271,sal,0.01670,5
5,liliana fuchs,271,azucar,0.01602,6
6,liliana fuchs,271,bien,0.01600,7
7,liliana fuchs,271,minutos,0.01550,8
8,liliana fuchs,271,queso,0.01497,9
9,liliana fuchs,271,limon,0.01468,10


## Reconocimiento de entidades y huellas geográficas

El objetivo aquí no es etiquetar personas, sino detectar entidades mencionadas por el corpus que puedan introducir sesgo geográfico o de estilo editorial.

En esta sección se usa spaCy con el modelo `es_core_news_sm`. Si el modelo no está instalado en el entorno, el bloque informa cómo añadirlo y no rompe el cuaderno.

In [45]:
NER_AVAILABLE = False
nlp = None
try:
    import spacy
except ImportError as exc:
    print(f'NER no disponible porque falta spacy: {exc}')
else:
    try:
        nlp = spacy.load('es_core_news_sm')
        NER_AVAILABLE = True
        print('Modelo spaCy es_core_news_sm cargado correctamente.')
    except OSError as exc:
        print('No se pudo cargar es_core_news_sm. Instala el modelo con:')
        print('python -m spacy download es_core_news_sm')
        print(f'Detalle: {exc}')

if NER_AVAILABLE and nlp is not None:
    sample_texts = []
    if author_col:
        sample_texts.extend(df[author_col].fillna('').astype(str).head(30).tolist())
    sample_texts.extend(df['titulo'].fillna('').astype(str).head(30).tolist())
    if 'instrucciones' in df.columns:
        sample_texts.extend(df['instrucciones'].fillna('').astype(str).sample(min(20, len(df)), random_state=42).tolist())

    entity_counts = Counter()
    entity_label_counter = Counter()
    entity_examples = defaultdict(list)

    for text in sample_texts:
        cleaned_text = str(text).strip()
        if not cleaned_text:
            continue
        doc = nlp(cleaned_text[:2000])
        for ent in doc.ents:
            label = ent.label_
            entity_text = normalize_text(ent.text)
            if not entity_text:
                continue
            entity_counts[(label, entity_text)] += 1
            entity_label_counter[label] += 1
            if len(entity_examples[label]) < 8:
                entity_examples[label].append(ent.text.strip())

    if entity_counts:
        rows = []
        for (label, entity_text), count in entity_counts.most_common(25):
            rows.append({'label': label, 'entidad': entity_text, 'frecuencia': count})
        ner_df = pd.DataFrame(rows)
        display(ner_df)
        print('Conteo de etiquetas NER:')
        print(entity_label_counter)
        print('Ejemplos por tipo de entidad:')
        for label, examples in entity_examples.items():
            print(f'- {label}: {sorted(set(examples))[:10]}')
    else:
        print('No se detectaron entidades en la muestra analizada.')
else:
    print('Se omite el análisis NER porque spaCy o el modelo es_core_news_sm no están disponibles en este entorno.')

No se pudo cargar es_core_news_sm. Instala el modelo con:
python -m spacy download es_core_news_sm
Detalle: [E050] Can't find model 'es_core_news_sm'. It doesn't seem to be a Python package or a valid path to a data directory.
Se omite el análisis NER porque spaCy o el modelo es_core_news_sm no están disponibles en este entorno.


## Interpretación cualitativa del NER

Cuando el NER devuelve entidades repetidas como lugares, marcas, nombres propios o referencias editoriales, normalmente está señalando dónde el corpus concentra su contexto externo. Si aparecen muchas entidades de tipo `LOC` o `GPE`, el corpus puede estar sesgado hacia ciertas geografías; si predominan `PERSON` u `ORG`, suele indicar peso de autores, blogs o firmas concretas.

En un corpus de recetas, este resultado no implica un sesgo demográfico clásico, sino un sesgo de procedencia y de cobertura: el sistema aprende mejor las referencias que se repiten en títulos, instrucciones o firmas, y puede dejar en segundo plano cocinas, regiones o estilos menos nombrados. Por eso el NER complementa al LDA: el LDA muestra concentración temática y el NER muestra concentración de entidades y contextos nominales.

## Resumen de sesgos observados

La lectura del corpus no debe quedarse solo en los numeros. A partir de las tablas anteriores conviene redactar una interpretacion orientada al sistema RAG:

- **Concentracion de fuentes o autores**: si una fuente domina, el asistente tendra un estilo menos diverso y repetira patrones editorialmente similares.
- **Sesgo tematico**: si ciertos terminos o grupos aparecen de forma desproporcionada, el RAG tendera a proponer recetas del mismo tipo antes que alternativas.
- **Sesgo geografico o de estilo culinario**: si aparecen muchas referencias a una cocina concreta, las respuestas cubriran peor otras tradiciones.
- **Sesgo de complejidad**: si abundan recetas muy rapidas o muy elaboradas, el asistente puede subrepresentar el otro extremo.

Estas observaciones deben incorporarse en la memoria del proyecto y en la discusion del sistema RAG, porque afectan tanto a la recuperacion como a la generacion.

## Medidas de mitigacion

Si el analisis confirma sesgos, las medidas recomendadas para este proyecto son:

1. **Diversificar la recopilacion**: ampliar dominios y blogs para que una sola fuente no monopolice el indice.
2. **Limitar la repeticion por dominio o autor**: aplicar muestreo estratificado o caps por fuente para evitar sobrepeso editorial.
3. **Anotar metadatos de procedencia**: guardar dominio, tipo de cocina, region y nivel de dificultad para poder filtrar y evaluar por segmentos.
4. **Deduplicar variantes cercanas**: usar similitud semantica o textual en lugar de depender solo de duplicados exactos.
5. **Evaluar por subgrupos**: medir retrieval y respuestas separadamente por cocina, fuente y complejidad.
6. **Reequilibrar el retrieval**: si una fuente domina, penalizar repeticiones o re-rankear para aumentar diversidad.
7. **Separar corpus y memoria de respuesta**: no asumir que una cocina o estilo dominante representa todo el espacio culinario.

En un RAG, estas medidas son tan importantes como el modelo generativo, porque el sesgo suele entrar por el indice y no solo por el LLM.

In [38]:
summary_rows = []
if author_col is not None:
    author_clean = df[author_col].fillna('').map(normalize_text)
    author_clean = author_clean[author_clean.astype(bool)]
    counts = author_clean.value_counts()
    summary_rows.append({'dimensiones': 'autor', 'n_grupos': int(counts.size), 'gini': round(gini_from_counts(counts.values), 3), 'hhi': round(hhi_from_counts(counts.values), 3)})
if source_col is not None:
    if 'url' in source_col.lower():
        source_clean = df[source_col].map(lambda x: re.sub(r'^https?://', '', normalize_text(x)).split('/')[0] if normalize_text(x) else '')
    else:
        source_clean = df[source_col].fillna('').map(normalize_text)
    source_clean = source_clean[source_clean.astype(bool)]
    counts = source_clean.value_counts()
    summary_rows.append({'dimensiones': 'fuente', 'n_grupos': int(counts.size), 'gini': round(gini_from_counts(counts.values), 3), 'hhi': round(hhi_from_counts(counts.values), 3)})
if time_col is not None:
    time_bucket_counts = df[time_col].map(length_bucket).value_counts()
    summary_rows.append({'dimensiones': 'tiempo', 'n_grupos': int(time_bucket_counts.size), 'gini': round(gini_from_counts(time_bucket_counts.values), 3), 'hhi': round(hhi_from_counts(time_bucket_counts.values), 3)})

summary_df = pd.DataFrame(summary_rows)
if not summary_df.empty:
    display(summary_df)
    summary_path = OUTPUT_DIR / 'sesgos_resumen.csv'
    summary_df.to_csv(summary_path, index=False)
    print(f'Resumen guardado en: {summary_path}')
else:
    print('No hubo metadatos suficientes para un resumen cuantitativo.')

,dimensiones,n_grupos,gini,hhi
0,autor,26,0.655,0.11
1,tiempo,4,0.256,0.31


Resumen guardado en: c:\Users\pc\Documents\MASTER\SEGUNDO SEMESTRE\NLP\Proyecto\ProyectoPLN\data\sesgos\sesgos_resumen.csv
